# Loading dataset

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('corporate_ai_adoption_dataset.csv')

# Target Variable Feature Engineering

In [3]:
# Create total financial benefit (Our absolute target variable)
df['total_financial_benefit'] = df['cost_savings'] + df['revenue_impact']

# Calculate raw ROI percentage for EDA metrics 
# Replacing 0 with 1 protects the line from crashing if a company invested $0
df['roi_percentage'] = (df['total_financial_benefit'] / df['ai_investment_usd'].replace(0, 1)) * 100

print("Target features created. Sample mapping:")
print(df[['cost_savings', 'revenue_impact', 'total_financial_benefit', 'roi_percentage']].head())

Target features created. Sample mapping:
   cost_savings  revenue_impact  total_financial_benefit  roi_percentage
0       3519354         2751513                  6270867       53.381633
1        295244          304776                   600020       47.349353
2       2472329         5170304                  7642633       93.557092
3        512061         -233448                   278613       22.573264
4       2122064         2110644                  4232708       84.643241


# Check for Outliers

In [4]:
# Define the numerical columns we want to inspect
columns_to_check = ['ai_investment_usd', 'employee_ai_training_hours', 'deployment_count', 'total_financial_benefit']

for col in columns_to_check:
    # 1. Calculate Quartiles (25th and 75th percentiles)
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    
    # 2. Compute Interquartile Range
    iqr = q3 - q1
    
    # 3. Establish mathematical boundaries
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    # 4. Count observations sitting outside the boundaries
    lower_outliers = df[df[col] < lower_bound].shape[0]
    upper_outliers = df[df[col] > upper_bound].shape[0]
    
    print(f"   Column: '{col}'")
    print(f"   - Normal Range: ${lower_bound:,.2f} to ${upper_bound:,.2f}")
    print(f"   - Min Observed: ${df[col].min():,.2f} | Max Observed: ${df[col].max():,.2f}")
    print(f"   - Outliers Detected: {lower_outliers} below floor | {upper_outliers} above ceiling")
    print("-"*50)

   Column: 'ai_investment_usd'
   - Normal Range: $-4,223,766.25 to $12,962,949.75
   - Min Observed: $47,888.00 | Max Observed: $54,170,345.00
   - Outliers Detected: 0 below floor | 7476 above ceiling
--------------------------------------------------
   Column: 'employee_ai_training_hours'
   - Normal Range: $-42.60 to $195.00
   - Min Observed: $1.00 | Max Observed: $197.90
   - Outliers Detected: 0 below floor | 6 above ceiling
--------------------------------------------------
   Column: 'deployment_count'
   - Normal Range: $-14.00 to $66.00
   - Min Observed: $3.00 | Max Observed: $58.00
   - Outliers Detected: 0 below floor | 0 above ceiling
--------------------------------------------------
   Column: 'total_financial_benefit'
   - Normal Range: $-5,979,154.88 to $13,113,694.12
   - Min Observed: $-9,070,236.00 | Max Observed: $138,010,586.00
   - Outliers Detected: 3 below floor | 14294 above ceiling
--------------------------------------------------


# Outlier Mitigation (99th Percentile Winsorization)

In [5]:
# Financial columns have massive enterprise variances, so we clamp them.
cols_to_clamp = ['ai_investment_usd', 'employee_ai_training_hours', 'total_financial_benefit']

# Cap extreme anomalies at 1% and 99% percentiles to stabilize tree splits
for col in cols_to_clamp:
    lower_bound = df[col].quantile(0.01)
    upper_bound = df[col].quantile(0.99)
    df[col] = np.clip(df[col], lower_bound, upper_bound)

print("Outlier boundaries successfully clamped using 99th percentile Winsorization.")

Outlier boundaries successfully clamped using 99th percentile Winsorization.


# Setting X and y

In [6]:
# Drop unique IDs, components of the answer, and derivative ratios to stop data leakage
X = df.drop(columns=[
    'company_id', 
    'cost_savings', 
    'revenue_impact', 
    'total_financial_benefit', 
    'roi_percentage'
])

y = df['total_financial_benefit']

# One-Hot Encoding

In [7]:
# Convert categorical columns (industry, country) into binary matrix columns
# drop_first=True reduces multi-collinearity issues
X_encoded = pd.get_dummies(X, columns=['industry', 'country'], drop_first=True)

print(f"Feature matrix isolated. Input feature space dimensions: {X_encoded.shape[1]} columns.")

Feature matrix isolated. Input feature space dimensions: 31 columns.


# Train-Test Split (80/20 Partition)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

print("Data successfully partitioned:")
print(f"  - X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"  - X_test:  {X_test.shape}  | y_test:  {y_test.shape}")

Data successfully partitioned:
  - X_train: (160000, 31) | y_train: (160000,)
  - X_test:  (40000, 31)  | y_test:  (40000,)


# Running the Model Tournament

In [9]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

print("Initializing models...")

# Tuned parameters: depth=6 keeps XGBoost from overfitting the 200k row scale
xgb_engine = XGBRegressor(
    n_estimators=150, 
    learning_rate=0.08, 
    max_depth=6, 
    random_state=42, 
    n_jobs=-1
)

# Parallel forest limits max depth to 12 to maintain generalizability
rf_engine = RandomForestRegressor(
    n_estimators=100, 
    max_depth=12, 
    random_state=42, 
    n_jobs=-1
)

print("Training XGBoost model layout...")
xgb_engine.fit(X_train, y_train)

print("Training Random Forest model layout...")
rf_engine.fit(X_train, y_train)

print("All tournament pipelines trained successfully.")

Initializing models...
Training XGBoost model layout...
Training Random Forest model layout...
All tournament pipelines trained successfully.


# Performance Evaluation & Serializing the Winner

In [10]:
import pickle
from sklearn.metrics import mean_absolute_error, r2_score

print("=== TOURNAMENT METRICS COMPARISON ===")

engines = {
    'XGBoost Regressor Engine': xgb_engine,
    'Random Forest Regressor Engine': rf_engine
}

top_r2 = -1
champion_model = None
champion_name = ""

# Loop through and score models against hidden validation data
for name, model in engines.items():
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    print(f"\n {name}:")
    print(f"   - R2 Score (Accuracy):        {r2:.5f}")
    print(f"   - Mean Absolute Error (MAE):  ${mae:,.2f}")
    
    # Save the highest performer dynamically
    if r2 > top_r2:
        top_r2 = r2
        champion_model = model
        champion_name = name

print("\n" + "="*40)
print(f" PRODUCTION CHAMPION: {champion_name} with R2 = {top_r2:.5f}")
print("="*40)

# Serialize the winning framework to disk for backend API usage
production_file = "champion_roi_model.pkl"
with open(production_file, 'wb') as file:
    pickle.dump(champion_model, file)

print(f"Artifact compiled and exported as '{production_file}'. Ready for backend routing.")

=== TOURNAMENT METRICS COMPARISON ===

 XGBoost Regressor Engine:
   - R2 Score (Accuracy):        0.84421
   - Mean Absolute Error (MAE):  $1,115,139.12

 Random Forest Regressor Engine:
   - R2 Score (Accuracy):        0.84270
   - Mean Absolute Error (MAE):  $1,120,982.93

 PRODUCTION CHAMPION: XGBoost Regressor Engine with R2 = 0.84421
Artifact compiled and exported as 'champion_roi_model.pkl'. Ready for backend routing.
